In [4]:
import sqlite3
import pandas as pd

raw_df = pd.read_csv("../data/telco_churn.csv")

print(f"Dataset loaded successfully with shape: {raw_df.shape}\n")
print(f"Columns in dataset: \n{list(raw_df.columns)}\n")

conn = sqlite3.connect("../data/telecom_finance.db")

raw_df.to_sql("raw_customer_metrics", conn, if_exists="replace", index=False)

print("Table 'raw_customer_metrics' successfully created in telecom_finance.db!")
conn.close()

Dataset loaded successfully with shape: (7043, 21)

Columns in dataset: 
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Table 'raw_customer_metrics' successfully created in telecom_finance.db!


In [6]:
conn = sqlite3.connect("../data/telecom_finance.db")

sql_cohort_query = """
WITH CleanedData AS (
    SELECT 
        customerID,
        gender,
        SeniorCitizen,
        Partner,
        Dependents,
        tenure,
        PhoneService,
        MultipleLines,
        InternetService,
        OnlineSecurity,
        OnlineBackup,
        DeviceProtection,
        TechSupport,
        StreamingTV,
        StreamingMovies,
        Contract,
        PaperlessBilling,
        PaymentMethod,
        CAST(MonthlyCharges AS REAL) AS MonthlyCharges,
        CAST(COALESCE(NULLIF(TRIM(TotalCharges), ''), 0) AS REAL) AS TotalCharges,
        Churn
    FROM raw_customer_metrics
),
TenureCohortData AS (
    SELECT 
        *,
        -- Customer Lifetime Tenure Bucketing
        CASE 
            WHEN tenure <= 12 THEN '0-12 Months'
            WHEN tenure > 12 AND tenure <= 24 THEN '12-24 Months'
            WHEN tenure > 24 AND tenure <= 48 THEN '24-48 Months'
            WHEN tenure > 48 AND tenure <= 60 THEN '48-60 Months'
            ELSE '60+ Months'
        END AS Tenure_Cohort,
        
        -- Window Function: Rank customers by Monthly Charges within each Contract bucket
        DENSE_RANK() OVER (
            PARTITION BY Contract 
            ORDER BY MonthlyCharges DESC
        ) AS Contract_Revenue_Rank
    FROM CleanedData
)
SELECT * FROM TenureCohortData;
"""

df_master = pd.read_sql_query(sql_cohort_query, conn)
conn.close()

print(f"Master Analytical DataFrame Shape: {df_master.shape}\n")
print(df_master[['customerID', 'Contract', 'tenure', 'Tenure_Cohort', 'MonthlyCharges', 'TotalCharges', 'Contract_Revenue_Rank']].head())

df_master.to_csv("../data/df_master_cleaned.csv", index=False)

Master Analytical DataFrame Shape: (7043, 23)

   customerID        Contract  tenure Tenure_Cohort  MonthlyCharges  \
0  2302-ANTDP  Month-to-month      48  24-48 Months          117.45   
1  8016-NCFVO  Month-to-month      55  48-60 Months          116.50   
2  9659-QEQSY  Month-to-month      45  24-48 Months          115.65   
3  4361-BKAXE  Month-to-month      41  24-48 Months          114.50   
4  6710-HSJRD  Month-to-month      61    60+ Months          114.10   

   TotalCharges  Contract_Revenue_Rank  
0       5438.90                      1  
1       6382.55                      2  
2       5125.50                      3  
3       4527.45                      4  
4       7132.15                      5  
